In [2]:
import os
from glob import glob
import datetime
from datetime import datetime, timezone

import folium
import geopandas as gpd
import h5py
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objs as go
import seaborn as sns
import xarray as xr
from folium.plugins import Draw
from plotly.subplots import make_subplots
from scipy.stats import skew, kurtosis, t
from shapely import geometry
import streamlit as st
from streamlit_folium import st_folium

ModuleNotFoundError: No module named 'plotting.plot_EFD'

In [ ]:
def plot_EFD(path):
    f = xr.open_dataset(path, engine = 'h5netcdf', phony_dims='sort')
    X_Waveform = f['A111_W'][...]
    Y_Waveform = f['A112_W'][...]
    Z_Waveform = f['A113_W'][...]
    verse_time = f['VERSE_TIME'][...].values.flatten()
    X_Power_spectrum = f['A111_P'][...]
    Y_Power_spectrum = f['A112_P'][...]
    Z_Power_spectrum = f['A113_P'][...]
    latitude = f['MAG_LAT'][...]
    longitude = f['MAG_LON'][...]
    frequency = f['FREQ'][...]
    magnitude = np.sqrt(X_Waveform**2 + Y_Waveform**2 + Z_Waveform**2)
    polar_angle = np.arccos(Z_Waveform / magnitude)  # theta
    azimuthal_angle = np.arctan2(Y_Waveform, X_Waveform)  # phi
 
    # Convert angles to degrees
    polar_angle = np.degrees(polar_angle)
    azimuthal_angle = np.degrees(azimuthal_angle)
 
    reduced_freq = 100  # Specify the desired frequency here
    X_Waveform = reduce_frequency(X_Waveform, reduced_freq)
    Y_Waveform = reduce_frequency(Y_Waveform, reduced_freq)
    Z_Waveform = reduce_frequency(Z_Waveform, reduced_freq)
    magnitude = reduce_frequency(magnitude, reduced_freq)
    polar_angle = reduce_frequency(polar_angle, reduced_freq)
    azimuthal_angle = reduce_frequency(azimuthal_angle, reduced_freq)
 
    X_Waveform = X_Waveform.values.flatten()
    Y_Waveform = Y_Waveform.values.flatten()
    Z_Waveform = Z_Waveform.values.flatten()
    magnitude = magnitude.values.flatten()
    polar_angle = polar_angle.values.flatten()
    azimuthal_angle = azimuthal_angle.values.flatten()
 
    len_time = len(verse_time)
 
    # Ensure vers_extend has the same length as the waveforms
    vers_extend = np.concatenate([np.linspace(verse_time[i], verse_time[i+1], reduced_freq, endpoint=False) for i in range(len_time-1)])
    vers_extend = np.concatenate([vers_extend, np.linspace(verse_time[-2], verse_time[-1], reduced_freq)])
    # First figure for waveforms and magnitude
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=vers_extend, y=X_Waveform, mode='lines', name='X Waveform'))
    fig1.add_trace(go.Scatter(x=vers_extend, y=Y_Waveform, mode='lines', name='Y Waveform'))
    fig1.add_trace(go.Scatter(x=vers_extend, y=Z_Waveform, mode='lines', name='Z Waveform'))
    fig1.add_trace(go.Scatter(x=vers_extend, y=magnitude, mode='lines', name='Vector sum'))
 
    fig1.update_layout(
        title="X, Y, Z Waveforms and the Vector Sum",
        xaxis_title="Time (ms)",
        yaxis_title="V/m",
        legend=dict(x=1, y=0.5)
    )
 
    # Second figure for polar and azimuthal angles
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=vers_extend, y=polar_angle, mode='lines', name='Polar Angle', yaxis='y2'))
    fig2.add_trace(go.Scatter(x=vers_extend, y=azimuthal_angle, mode='lines', name='Azimuthal Angle', yaxis='y3'))
 
    fig2.update_layout(
        title="Polar and Azimuthal Angles",
        xaxis_title="Time (ms)",
        yaxis2=dict(
            title="Polar Angle (degrees)",
            range=[0, 180],
            side='left'
        ),
        yaxis3=dict(
            title="Azimuthal Angle (degrees)",
            range=[0, 360],
            overlaying='y2',
            side='right'
        ),
        legend=dict(x=1.08, y=0.54)
    )
 
    # Third figure for power spectra
    power_spectra = {'X': X_Power_spectrum, 'Y': Y_Power_spectrum, 'Z': Z_Power_spectrum}
    
    # for axis, spectrum in power_spectra.items():
    #     fig = go.Figure()
    #     fig.add_trace(go.Heatmap(
    #         z=np.log10(spectrum.values.T),  # Apply log10 to intensity (z-axis)
    #         x=verse_time,
    #         y=frequency,
    #         colorscale='Viridis',
    #         colorbar=dict(title='Log Power Spectrum')
    #     ))
    #     fig.update_layout(
    #         title=f"{axis} Power Spectrum",
    #         xaxis_title="Time (ms)",
    #         yaxis_title="Frequency (Hz)",
    #         legend=dict(x=1, y=0.5)
    #     )
        
    #     st.plotly_chart(fig)
 
    # columns = st.columns(len(power_spectra))
    # column_widths = [3, 3]
    # columns = st.columns(column_widths)
    
    # for i, (axis, spectrum) in enumerate(power_spectra.items()):
    #     with columns[i]:
    #         fig = go.Figure()
    #         fig.add_trace(go.Heatmap(
    #             z=np.log10(spectrum.values.T),  # Apply log10 to intensity (z-axis)
    #             x=verse_time,
    #             y=frequency,
    #             colorscale='Viridis',
    #             colorbar=dict(title='Log Power Spectrum')
    #         ))
    #         fig.update_layout(
    #             title=f"{axis} Power Spectrum",
    #             xaxis_title="Time (ms)",
    #             yaxis_title="Frequency (Hz)",
    #             legend=dict(x=1, y=0.5)
    #         )
            
    #         st.plotly_chart(fig)

    columns = st.columns(2)

    for i, (axis, spectrum) in enumerate(power_spectra.items()):
        if i < 2:
            with columns[i]:
                fig = go.Figure()
                fig.add_trace(go.Heatmap(
                    z=np.log10(spectrum.T),  # Apply log10 to intensity (z-axis)
                    x=verse_time,
                    y=frequency,
                    colorscale='Viridis',
                    colorbar=dict(title='Log Power Spectrum')
                ))
                fig.update_layout(
                    title=f"{axis} Power Spectrum",
                    xaxis_title="Time (ms)",
                    yaxis_title="Frequency (Hz)",
                    legend=dict(x=1, y=0.5)
                )
                st.plotly_chart(fig)

    # Display the third graph in a new row, centered
    if len(power_spectra) > 2:
        empty_col1, centered_col, empty_col2 = st.columns([1, 2, 1])
        with centered_col:
            fig = go.Figure()
            spectrum = list(power_spectra.values())[2]
            axis = list(power_spectra.keys())[2]
            fig.add_trace(go.Heatmap(
                z=np.log10(spectrum.T),  # Apply log10 to intensity (z-axis)
                x=verse_time,
                y=frequency,
                colorscale='Viridis',
                colorbar=dict(title='Log Power Spectrum')
            ))
            fig.update_layout(
                title=f"{axis} Power Spectrum",
                xaxis_title="Time (ms)",
                yaxis_title="Frequency (Hz)",
                legend=dict(x=1, y=0.5)
            )
            st.plotly_chart(fig)

    # Display the first two figures
    col1, col2 = st.columns(2)

    with col1:
        st.plotly_chart(fig1)

    with col2:
        st.plotly_chart(fig2)

In [1]:
from plotting import plot_EFD
from plotting.plot_LAP import lap_plot
from plotting.plot_HEPPX import heppx_plot
from plotting.plot_SCM import scmplot

ModuleNotFoundError: No module named 'plotting_CSES'

In [1]:
#REPLACE
folder_path = '/home/wvuser/scientific-dashboard/dataCSES'

def dataset(path):
    return xr.open_dataset(path, engine = 'h5netcdf', phony_dims = 'sort')
 
# Function to list all variable names in a dataset
def variables(data):
    return list(data.keys())

# Function to calculate leftmost and rightmost latitude extremes
def calculate_extremes(coordinates):
    if coordinates:
        min_lat = min(point[1] for point in coordinates[0])
        max_lat = max(point[1] for point in coordinates[0])
        return min_lat, max_lat
    return None, None

# Function for basic statistical analysis and plot
# def basic_statistical_analysis_and_plot(ds, sensor_names, selected_stats):
#     stats = {}
#     stats_functions = {
#         'mean': np.nanmean,
#         'median': np.nanmedian,
#         'std_dev': np.nanstd,
#         'min': np.nanmin,
#         'max': np.nanmax,
#         'variance': np.nanvar,
#         'skewness': lambda x: skew(x, nan_policy='omit'),
#         'kurtosis': lambda x: kurtosis(x, nan_policy='omit'),
#         'conf_interval_low': lambda x: t.interval(0.95, len(x)-1, loc=np.nanmean(x), scale=np.nanstd(x)/np.sqrt(len(x)))[0],
#         'conf_interval_high': lambda x: t.interval(0.95, len(x)-1, loc=np.nanmean(x), scale=np.nanstd(x)/np.sqrt(len(x)))[1]
#     }

#     for sensor_name in sensor_names:
#         if sensor_name in ds.data_vars:
#             data = ds[sensor_name].values.flatten()  # Flatten array

#             if np.issubdtype(data.dtype, np.number):
#                 data = data[np.isfinite(data)]  # Remove infinite values
#                 stats[sensor_name] = {stat: stats_functions[stat](data) for stat in selected_stats}
#             else:
#                 st.write(f"Variable {sensor_name} is not numeric or has unsupported data type.")
#         else:
#             st.write(f"Variable {sensor_name} is not found in the dataset.")

#     for sensor_name, stat_values in stats.items():
#         stats_df = pd.DataFrame(stat_values, index=[0])
#         fig, ax = plt.subplots(figsize=(10, 6))
#         sns.set(font_scale=1.2)
#         heatmap = sns.heatmap(stats_df, annot=True, cmap='viridis', fmt='.2f', linewidths=0.5, linecolor='white', cbar=True, annot_kws={"size": 8}, ax=ax)
#         ax.set_title(f'Statistical Metrics Heatmap for {sensor_name}', fontsize=16)
#         plt.xticks(rotation=45)
#         plt.yticks(rotation=0)
#         plt.tight_layout()
#         st.pyplot(fig)
#         plt.close(fig)

#     return stats


# Function for map drawing
def draw_map():
    # print('asdasdakjsdnas')
    m = folium.Map(location=[45.5236, -122.6750], zoom_start=1.15)
    draw = Draw(
        draw_options={
            'polyline': False,
            'polygon': True,
            'circle': False,
            'marker': False,
            'circlemarker': False,
            'rectangle': True,
        }
    )
    draw.add_to(m)
    st_map = st_folium(m, width=1000, height=400)

    last_active_drawing = st_map.get('last_active_drawing')
    if last_active_drawing:
        geometry = last_active_drawing.get('geometry')
        if geometry and 'coordinates' in geometry:
            coordinates = geometry['coordinates']
            # st.write("Coordinates of the last active drawing:")
            # for point in coordinates[0]:  # Assuming it's a polygon and accessing the outer ring
            #     lng, lat = point
            #     st.write(f"Longitude: {lng}, Latitude: {lat}")
            
            coords = geometry['coordinates'][0]  # Access the first inner list
       
            coords = coords[:-1]

            #call function and pass coords and file path
            pathskidibidi = polygon(coords, file_selector())

            dataset_type = extract_dataset_type(pathskidibidi)
            print(dataset_type)
            st.write("ajdnajdkansjkdna")
            st.write(dataset_type)

            option = st.multiselect(
                "Instrument Type",
                (" ", "EFD", "SCM", "LAP", "HEP1", "HEPX"))

            if st.button("plot"):
                if option:
                    for dataset in dataset_type:
                        file_path = dataset[0]
                        dataset_type = dataset[1]
                        for sensors in option:
                            if dataset_type == 'EFD' == sensors:
                                plot_EFD(file_path)
                            elif dataset_type == 'LAP' == sensors:
                                lap_plot(file_path)
                            elif dataset_type == 'HEPX' == sensors:
                                heppx_plot(file_path)
                            elif dataset_type == 'HEP1' == sensors:
                                st.write("working on function")
                            elif dataset_type == 'SCM' == sensors:
                                scmplot(file_path)
            min_lat, max_lat = calculate_extremes(coordinates)
            st.write(f"Leftmost Latitude: {min_lat:.6f}")
            st.write(f"Rightmost Latitude: {max_lat:.6f}")


        else:
            st.write("No 'coordinates' found in 'geometry'.")
    else:
        st.write("No 'last_active_drawing' found in st_map.")

# Function for statistical analysis
# def statistical_analysis():
#     st.header("Statistical Analysis and Visualization")
#     # folder_path = st.text_input("Enter the folder path:", '/home/wvuser/project/data')
#     if folder_path:
#         file_path = file_selector()
#         if not os.path.isfile(file_path):
#             st.error(f"Error: File {file_path} does not exist.")
#             return

#         try:
#             ds = xr.open_dataset(file_path, engine='h5netcdf', phony_dims='sort')
#             st.write("Dataset loaded successfully.")
#         except Exception as e:
#             st.error(f"Error opening file: {e}")
#             return

#         available_sensors = list(ds.data_vars.keys())
#         selected_sensors = st.multiselect("Select sensor names to analyze:", available_sensors)

#         available_stats = [
#             'mean', 'median', 'std_dev', 'min', 'max', 'variance', 
#             'skewness', 'kurtosis', 'conf_interval_low', 'conf_interval_high'
#         ]
#         selected_stats = st.multiselect("Select statistics to calculate:", available_stats)

#         if st.button("Analyze"):
#             if not selected_sensors:
#                 st.error("Please select at least one sensor.")
#             elif not selected_stats:
#                 st.error("Please select at least one statistical calculation.")
#             else:
#                 stats = basic_statistical_analysis_and_plot(ds, selected_sensors, selected_stats)
#                 if stats:
#                     for sensor, stat_values in stats.items():
#                         st.write(f"Statistics for {sensor}:")
#                         for key, value in stat_values.items():
#                             st.write(f"{key}: {value}")
#                         st.write("")


# def filter_data_within_coordinates(coordinates, file_path):
#     try:
#         ds = xr.open_dataset(file_path, engine='h5netcdf', phony_dims='sort')
#         st.write("Dataset loaded successfully.")
#     except Exception as e:
#         st.error(f"Error opening file: {e}")
#         return None

#     # Ensure the dataset contains GEO_LAT and GEO_LON
#     if 'GEO_LAT' not in ds or 'GEO_LON' not in ds:
#         st.error("Dataset does not contain GEO_LAT and GEO_LON variables.")
#         return None

#     geo_lat = ds['GEO_LAT']
#     geo_lon = ds['GEO_LON']

#     # Extract latitude and longitude boundaries from coordinates
#     min_lon = min(point[0] for point in coordinates)
#     max_lon = max(point[0] for point in coordinates)
#     min_lat = min(point[1] for point in coordinates)
#     max_lat = max(point[1] for point in coordinates)

#     # Filter data within the coordinate boundaries
#     lat_filter = (geo_lat >= min_lat) & (geo_lat <= max_lat)
#     lon_filter = (geo_lon >= min_lon) & (geo_lon <= max_lon)
#     combined_filter = lat_filter & lon_filter

#     # Convert the combined filter to a DataArray
#     combined_filter_da = xr.DataArray(combined_filter, dims=geo_lat.dims, coords=geo_lat.coords)

#     # Apply filter to dataset
#     filtered_ds = ds.where(combined_filter_da, drop=True)

#     # Check if there are any valid data points left after filtering
#     if filtered_ds.sizes['time'] == 0:  # Adjust 'time' dimension if necessary
#         st.error("No data points found within the specified coordinates.")
#         return None

#     return filtered_ds



def polygon(points, files):    
    intersectionFile = []
 
    latitudes = [point[1] for point in points]
    longitudes = [point[0] for point in points]

    latitudes.pop()
    longitudes.pop()
 
    lat_min = min(latitudes)
    lat_max = max(latitudes)
    lon_min = min(longitudes)
    lon_max = max(longitudes)
 
    for file in files:
 
        ds = dataset(file)
        geo_lat = ds.GEO_LAT
        geo_lon = ds.GEO_LON
 
        lat_mask = (geo_lat >= lat_min) & (geo_lat <= lat_max)
        lon_mask = (geo_lon >= lon_min) & (geo_lon <= lon_max)
 
        final_mask = lat_mask & lon_mask
 
        if final_mask.any():
            intersectionFile.append(file)
    return intersectionFile
 



def extract_dataset_type(file_paths):    
    dataset_types = []
    
    for file_path in file_paths:
        file_name = file_path.split('/')[-1]
        
        if 'EFD' in file_name:
            dataset_types.append([file_path, 'EFD'])
        elif 'LAP' in file_name:
            dataset_types.append([file_path, 'LAP'])
        elif 'SCM' in file_name:
            dataset_types.append([file_path, 'SCM'])
        elif 'HEP_4' in file_name:
            dataset_types.append([file_path, 'HEPX'])
        elif 'HEP_1' in file_name:
            dataset_types.append([file_path, 'HEP1'])

        else:
            raise ValueError(f"Unknown dataset type in file name: {file_name}")
    
    return dataset_types


def file_selector(folder_path='/home/wvuser/project/data/'):
    filenames = os.listdir(folder_path)
    file_paths = [os.path.join(folder_path, filename) for filename in filenames]    
    return file_paths

 

# Main function
def main():
    st.set_page_config(layout="wide")
    st.title("Map Drawing and Statistical Analysis Tool")
    draw_map()
    # statistical_analysis()

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'plotting.plot_EFD'

In [5]:
import os
from glob import glob
import datetime
from datetime import datetime, timezone

import folium
import geopandas as gpd
import h5py
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objs as go
import seaborn as sns
import xarray as xr
from folium.plugins import Draw
from plotly.subplots import make_subplots
from scipy.stats import skew, kurtosis, t
from shapely import geometry
import streamlit as st
from streamlit_folium import st_folium

from plotting.plot_EFD import plot_EFD
from plotting.plot_LAP import lap_plot
from plotting.plot_HEPPX import heppx_plot
from plotting.plot_SCM import scmplot
#REPLACE
folder_path = '/home/wvuser/scientific-dashboard/dataCSES'

def dataset(path):
    return xr.open_dataset(path, engine = 'h5netcdf', phony_dims = 'sort')
 
# Function to list all variable names in a dataset
def variables(data):
    return list(data.keys())

# Function to calculate leftmost and rightmost latitude extremes
def calculate_extremes(coordinates):
    if coordinates:
        min_lat = min(point[1] for point in coordinates[0])
        max_lat = max(point[1] for point in coordinates[0])
        return min_lat, max_lat
    return None, None

# Function for basic statistical analysis and plot
# def basic_statistical_analysis_and_plot(ds, sensor_names, selected_stats):
#     stats = {}
#     stats_functions = {
#         'mean': np.nanmean,
#         'median': np.nanmedian,
#         'std_dev': np.nanstd,
#         'min': np.nanmin,
#         'max': np.nanmax,
#         'variance': np.nanvar,
#         'skewness': lambda x: skew(x, nan_policy='omit'),
#         'kurtosis': lambda x: kurtosis(x, nan_policy='omit'),
#         'conf_interval_low': lambda x: t.interval(0.95, len(x)-1, loc=np.nanmean(x), scale=np.nanstd(x)/np.sqrt(len(x)))[0],
#         'conf_interval_high': lambda x: t.interval(0.95, len(x)-1, loc=np.nanmean(x), scale=np.nanstd(x)/np.sqrt(len(x)))[1]
#     }

#     for sensor_name in sensor_names:
#         if sensor_name in ds.data_vars:
#             data = ds[sensor_name].values.flatten()  # Flatten array

#             if np.issubdtype(data.dtype, np.number):
#                 data = data[np.isfinite(data)]  # Remove infinite values
#                 stats[sensor_name] = {stat: stats_functions[stat](data) for stat in selected_stats}
#             else:
#                 st.write(f"Variable {sensor_name} is not numeric or has unsupported data type.")
#         else:
#             st.write(f"Variable {sensor_name} is not found in the dataset.")

#     for sensor_name, stat_values in stats.items():
#         stats_df = pd.DataFrame(stat_values, index=[0])
#         fig, ax = plt.subplots(figsize=(10, 6))
#         sns.set(font_scale=1.2)
#         heatmap = sns.heatmap(stats_df, annot=True, cmap='viridis', fmt='.2f', linewidths=0.5, linecolor='white', cbar=True, annot_kws={"size": 8}, ax=ax)
#         ax.set_title(f'Statistical Metrics Heatmap for {sensor_name}', fontsize=16)
#         plt.xticks(rotation=45)
#         plt.yticks(rotation=0)
#         plt.tight_layout()
#         st.pyplot(fig)
#         plt.close(fig)

#     return stats


# Function for map drawing
def draw_map():
    # print('asdasdakjsdnas')
    m = folium.Map(location=[45.5236, -122.6750], zoom_start=1.15)
    draw = Draw(
        draw_options={
            'polyline': False,
            'polygon': True,
            'circle': False,
            'marker': False,
            'circlemarker': False,
            'rectangle': True,
        }
    )
    draw.add_to(m)
    st_map = st_folium(m, width=1000, height=400)

    last_active_drawing = st_map.get('last_active_drawing')
    if last_active_drawing:
        geometry = last_active_drawing.get('geometry')
        if geometry and 'coordinates' in geometry:
            coordinates = geometry['coordinates']
            # st.write("Coordinates of the last active drawing:")
            # for point in coordinates[0]:  # Assuming it's a polygon and accessing the outer ring
            #     lng, lat = point
            #     st.write(f"Longitude: {lng}, Latitude: {lat}")
            
            coords = geometry['coordinates'][0]  # Access the first inner list
       
            coords = coords[:-1]

            #call function and pass coords and file path
            pathskidibidi = polygon(coords, file_selector())

            dataset_type = extract_dataset_type(pathskidibidi)
            print(dataset_type)
            st.write("ajdnajdkansjkdna")
            st.write(dataset_type)

            option = st.multiselect(
                "Instrument Type",
                (" ", "EFD", "SCM", "LAP", "HEP1", "HEPX"))

            if st.button("plot"):
                if option:
                    for dataset in dataset_type:
                        file_path = dataset[0]
                        dataset_type = dataset[1]
                        for sensors in option:
                            if dataset_type == 'EFD' == sensors:
                                plot_EFD(file_path)
                            elif dataset_type == 'LAP' == sensors:
                                lap_plot(file_path)
                            elif dataset_type == 'HEPX' == sensors:
                                heppx_plot(file_path)
                            elif dataset_type == 'HEP1' == sensors:
                                st.write("working on function")
                            elif dataset_type == 'SCM' == sensors:
                                scmplot(file_path)
            min_lat, max_lat = calculate_extremes(coordinates)
            st.write(f"Leftmost Latitude: {min_lat:.6f}")
            st.write(f"Rightmost Latitude: {max_lat:.6f}")


        else:
            st.write("No 'coordinates' found in 'geometry'.")
    else:
        st.write("No 'last_active_drawing' found in st_map.")

# Function for statistical analysis
# def statistical_analysis():
#     st.header("Statistical Analysis and Visualization")
#     # folder_path = st.text_input("Enter the folder path:", '/home/wvuser/project/data')
#     if folder_path:
#         file_path = file_selector()
#         if not os.path.isfile(file_path):
#             st.error(f"Error: File {file_path} does not exist.")
#             return

#         try:
#             ds = xr.open_dataset(file_path, engine='h5netcdf', phony_dims='sort')
#             st.write("Dataset loaded successfully.")
#         except Exception as e:
#             st.error(f"Error opening file: {e}")
#             return

#         available_sensors = list(ds.data_vars.keys())
#         selected_sensors = st.multiselect("Select sensor names to analyze:", available_sensors)

#         available_stats = [
#             'mean', 'median', 'std_dev', 'min', 'max', 'variance', 
#             'skewness', 'kurtosis', 'conf_interval_low', 'conf_interval_high'
#         ]
#         selected_stats = st.multiselect("Select statistics to calculate:", available_stats)

#         if st.button("Analyze"):
#             if not selected_sensors:
#                 st.error("Please select at least one sensor.")
#             elif not selected_stats:
#                 st.error("Please select at least one statistical calculation.")
#             else:
#                 stats = basic_statistical_analysis_and_plot(ds, selected_sensors, selected_stats)
#                 if stats:
#                     for sensor, stat_values in stats.items():
#                         st.write(f"Statistics for {sensor}:")
#                         for key, value in stat_values.items():
#                             st.write(f"{key}: {value}")
#                         st.write("")


# def filter_data_within_coordinates(coordinates, file_path):
#     try:
#         ds = xr.open_dataset(file_path, engine='h5netcdf', phony_dims='sort')
#         st.write("Dataset loaded successfully.")
#     except Exception as e:
#         st.error(f"Error opening file: {e}")
#         return None

#     # Ensure the dataset contains GEO_LAT and GEO_LON
#     if 'GEO_LAT' not in ds or 'GEO_LON' not in ds:
#         st.error("Dataset does not contain GEO_LAT and GEO_LON variables.")
#         return None

#     geo_lat = ds['GEO_LAT']
#     geo_lon = ds['GEO_LON']

#     # Extract latitude and longitude boundaries from coordinates
#     min_lon = min(point[0] for point in coordinates)
#     max_lon = max(point[0] for point in coordinates)
#     min_lat = min(point[1] for point in coordinates)
#     max_lat = max(point[1] for point in coordinates)

#     # Filter data within the coordinate boundaries
#     lat_filter = (geo_lat >= min_lat) & (geo_lat <= max_lat)
#     lon_filter = (geo_lon >= min_lon) & (geo_lon <= max_lon)
#     combined_filter = lat_filter & lon_filter

#     # Convert the combined filter to a DataArray
#     combined_filter_da = xr.DataArray(combined_filter, dims=geo_lat.dims, coords=geo_lat.coords)

#     # Apply filter to dataset
#     filtered_ds = ds.where(combined_filter_da, drop=True)

#     # Check if there are any valid data points left after filtering
#     if filtered_ds.sizes['time'] == 0:  # Adjust 'time' dimension if necessary
#         st.error("No data points found within the specified coordinates.")
#         return None

#     return filtered_ds



def polygon(points, files):    
    intersectionFile = []
 
    latitudes = [point[1] for point in points]
    longitudes = [point[0] for point in points]

    latitudes.pop()
    longitudes.pop()
 
    lat_min = min(latitudes)
    lat_max = max(latitudes)
    lon_min = min(longitudes)
    lon_max = max(longitudes)
 
    for file in files:
 
        ds = dataset(file)
        geo_lat = ds.GEO_LAT
        geo_lon = ds.GEO_LON
 
        lat_mask = (geo_lat >= lat_min) & (geo_lat <= lat_max)
        lon_mask = (geo_lon >= lon_min) & (geo_lon <= lon_max)
 
        final_mask = lat_mask & lon_mask
 
        if final_mask.any():
            intersectionFile.append(file)
    return intersectionFile
 



def extract_dataset_type(file_paths):    
    dataset_types = []
    
    for file_path in file_paths:
        file_name = file_path.split('/')[-1]
        
        if 'EFD' in file_name:
            dataset_types.append([file_path, 'EFD'])
        elif 'LAP' in file_name:
            dataset_types.append([file_path, 'LAP'])
        elif 'SCM' in file_name:
            dataset_types.append([file_path, 'SCM'])
        elif 'HEP_4' in file_name:
            dataset_types.append([file_path, 'HEPX'])
        elif 'HEP_1' in file_name:
            dataset_types.append([file_path, 'HEP1'])

        else:
            raise ValueError(f"Unknown dataset type in file name: {file_name}")
    
    return dataset_types


def file_selector(folder_path='/home/wvuser/project/data/'):
    filenames = os.listdir(folder_path)
    file_paths = [os.path.join(folder_path, filename) for filename in filenames]    
    return file_paths

 

# Main function
def main():
    st.set_page_config(layout="wide")
    st.title("Map Drawing and Statistical Analysis Tool")
    draw_map()
    # statistical_analysis()

if __name__ == "__main__":
    main()

2024-07-01 15:33:32.079 
  command:

    streamlit run /Users/emilyzhao/sample-project/.venv/lib/python3.11/site-packages/ipykernel_launcher.py [ARGUMENTS]
2024-07-01 15:33:32.086 Session state does not function when running a script without `streamlit run`
